In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
diabetes = load_diabetes()

df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)

df['Target'] = diabetes.target

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
plt.figure(figsize=(10,7))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
plt.figure(figsize=(7,5))

sns.regplot(x=df['bmi'], y=df['Target'],
            line_kws={'color': 'red'})

plt.title("BMI vs Diabetes Progression")
plt.xlabel("BMI")
plt.ylabel("Target")

plt.show()

In [ ]:
X = df.drop('Target', axis=1)

Y = df['Target']

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

In [ ]:
lr = LinearRegression()

lr.fit(X_train, Y_train)

pred_lr = lr.predict(X_test)

In [ ]:
pr = make_pipeline(
    PolynomialFeatures(degree=2),
    StandardScaler(),
    LinearRegression()
)

pr.fit(X_train, Y_train)

pred_pr = pr.predict(X_test)

In [ ]:
ridge = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

ridge.fit(X_train, Y_train)

pred_ridge = ridge.predict(X_test)

In [ ]:
lasso = make_pipeline(
    StandardScaler(),
    Lasso(alpha=0.1)
)

lasso.fit(X_train, Y_train)

pred_lasso = lasso.predict(X_test)

In [ ]:
dt = DecisionTreeRegressor(random_state=42)

dt.fit(X_train, Y_train)

pred_dt = dt.predict(X_test)

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, Y_train)

pred_rf = rf.predict(X_test)

In [ ]:
models = {
    'Linear Regression': pred_lr,
    'Polynomial Regression': pred_pr,
    'Ridge Regression': pred_ridge,
    'Lasso Regression': pred_lasso,
    'Decision Tree': pred_dt,
    'Random Forest': pred_rf
}

results = []

for name, pred in models.items():

    mae = mean_absolute_error(Y_test, pred)

    mse = mean_squared_error(Y_test, pred)

    rmse = np.sqrt(mse)

    r2 = r2_score(Y_test, pred)

    results.append([name, mae, mse, rmse, r2])

result_df = pd.DataFrame(
    results,
    columns=['Model', 'MAE', 'MSE', 'RMSE', 'R2 Score']
)

result_df

In [ ]:
plt.figure(figsize=(6,5))

plt.scatter(Y_test, pred_rf)

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")

plt.title("Actual vs Predicted Values")

plt.show()

In [ ]:
#Understand overfitting and underfitting
train_score = []
test_score = []

model_names = []

for name, model in [
    ('Linear Regression', lr),
    ('Decision Tree', dt),
    ('Random Forest', rf)
]:

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_score.append(r2_score(Y_train, train_pred))
    test_score.append(r2_score(Y_test, test_pred))

    model_names.append(name)

comparison = pd.DataFrame({
    'Model': model_names,
    'Training Score': train_score,
    'Testing Score': test_score
})

comparison

In [ ]:
comparison.set_index('Model').plot(kind='bar', figsize=(8,5))

plt.title("Training vs Testing Performance")
plt.ylabel("R2 Score")
plt.xticks(rotation=0)
plt.show()